# Baseline RAG - Colab Runner
Thin runner notebook. Core logic stays in the Python modules.

## Clone private repo
Use a GitHub Personal Access Token with read access to this repo.

In [3]:
import getpass
import os
import subprocess

os.chdir('/content')

token = getpass.getpass('GitHub token (repo read access): ')
result = subprocess.run(
    ['git', 'clone', f'https://{token}@github.com/zahiTouqan/latent-rag.git'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    # Strip token from error output before printing
    print('STDOUT:', result.stdout.replace(token, '***'))
    print('STDERR:', result.stderr.replace(token, '***'))
    raise SystemExit('Clone failed')
else:
    print('Clone successful')

STDOUT: 
STDERR: fatal: destination path 'latent-rag' already exists and is not an empty directory.



SystemExit: Clone failed

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
%cd /content/latent-rag
!git fetch
!git checkout v2
!git pull origin v2
!python -m pip install -U pip
!pip install -r requirements.txt
!pip install "datasets>=2.14,<3.0"
!pip install -q --upgrade transformers

/content/latent-rag
Branch 'v2' set up to track remote branch 'v2' from 'origin'.
Switched to a new branch 'v2'
From https://github.com/zahiTouqan/latent-rag
 * branch            v2         -> FETCH_HEAD
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.4 MB/s eta 0:00:0000:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 7.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 37.1 MB/s  0:00:00m0:00:0100:01
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [faiss-cpu]
    Found existing installation: datasets 4.0.0━━━━━━━━━━━━━━━ 1/3 [faiss-cpu]
    Uninstalling datasets-4.0.0:━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [faiss-

In [ ]:
!git pull origin v2

From https://github.com/zahiTouqan/latent-rag
 * branch            v2         -> FETCH_HEAD
Already up to date.


## Download Some NQ data
Streams `facebook/dpr` NQs + related data so that we do not download all the questions on Colab. (Can be changed by modifying the number inside the square bracket)

In [ ]:
import json, gzip, urllib.request

url = "https://dl.fbaipublicfiles.com/dpr/data/retriever/biencoder-nq-dev.json.gz"
with urllib.request.urlopen(url) as response:
    with gzip.open(response, 'rt', encoding='utf-8') as f:
        data = json.load(f)

sample = data[:100]

Just an example to visualize what an individual sample looks like

In [ ]:
print("First training sample:")
import json
print(json.dumps(sample[0], indent=2))

First training sample:
{
  "dataset": "nq_dev_psgs_w100",
  "question": "who sings does he love me with reba",
  "answers": [
    "Linda Davis"
  ],
  "positive_ctxs": [
    {
      "title": "Does He Love You",
      "text": "Does He Love You \"Does He Love You\" is a song written by Sandy Knox and Billy Stritch, and recorded as a duet by American country music artists Reba McEntire and Linda Davis. It was released in August 1993 as the first single from Reba's album \"Greatest Hits Volume Two\". It is one of country music's several songs about a love triangle. \"Does He Love You\" was written in 1982 by Billy Stritch. He recorded it with a trio in which he performed at the time, because he wanted a song that could be sung by the other two members",
      "score": 1000,
      "title_score": 1,
      "passage_id": "11828866"
    },
    {
      "title": "Does He Love You",
      "text": "Does He Love You \"Does He Love You\" is a song written by Sandy Knox and Billy Stritch, and recorded

## Build mini Corpus based on the NQs
Each DPR sample has pre-chunked `positive_ctxs` (i.e. related documents) and `hard_negative_ctxs` (unrelated documents). We collect them from the data and build a mini corpus that we embed later (so that we do not download all documents on Colab :D)

In [ ]:
import json
from pathlib import Path

CORPUS_OUT = Path("data/passages.jsonl")
CORPUS_OUT.parent.mkdir(parents=True, exist_ok=True)

corpus = {}  # passage_id -> record

for item in sample:
    for p in item["positive_ctxs"] + item["hard_negative_ctxs"]:
        pid = p["passage_id"]
        if pid not in corpus:
            corpus[pid] = {
                "id": pid,
                "text": f"{p['title']}: {p['text']}",
                "doc_id": p["title"]   # article title as document-level grouping
            }

with CORPUS_OUT.open("w", encoding="utf-8") as f:
    for record in corpus.values():
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(corpus)} passages to {CORPUS_OUT}")

Saved 10010 passages to data/passages.jsonl


## Build the index
This creates `index.faiss`, `metadata.jsonl`, and `config.json`. Make sure this step returns True

In [ ]:
import torch
print(torch.cuda.is_available())  # should be True on T4

True


In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Safely get your token from the side panel
token = userdata.get('HF_TOKEN')

# Log HuggingFace library into your account
login(token)


TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

Now we create the FAISS index of the corpus (should not take >4 mins for 10k passages)

In [ ]:
INDEX_DIR = '/content/index'
!python build_index.py --corpus_path {CORPUS_OUT} --index_dir {INDEX_DIR} --max_docs 100 --batch_size 32

Loaded 100 passages from data/passages.jsonl
Loading embedding model (Encoder only): google/t5gemma-2b-2b-ul2
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 732/732 [00:01<00:00, 437.31it/s]
Embedding 100 passages (4 batches)...
  batch 4/4
Saved index to /content/index


In [ ]:
!ls -lh /content/index


total 64M
-rw-r--r-- 1 root root 195 May 19 10:27 config.json
-rw-r--r-- 1 root root 64M May 19 10:27 latents_0.safetensors
-rw-r--r-- 1 root root 67K May 19 10:27 metadata.jsonl


## Evaluate
We first need to build the eval file to compare the results to the ground truth. We build it from the DPR data. It is just a JSONL file with question, answer, and optionally relevant_ids fields.

In [ ]:
import json
from pathlib import Path

EVAL_OUT = Path("data/eval.jsonl")

with EVAL_OUT.open("w", encoding="utf-8") as f:
    for item in sample:
        record = {
            "question": item["question"],
            "answer": item["answers"],
            "relevant_ids": [p["title"] for p in item["positive_ctxs"]]
        }
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(sample)} eval examples to {EVAL_OUT}")

Saved 100 eval examples to data/eval.jsonl


Then we run the actual evaluation:

In [ ]:
!git pull origin v2

From https://github.com/zahiTouqan/latent-rag
 * branch            v2         -> FETCH_HEAD
Already up to date.


In [ ]:
!python3 evaluate.py \
    --index_dir /content/index \
    --eval_path data/eval.jsonl \
    --retrieval_id_field source_doc_id \
    --top_k 5


: 

In [ ]:
!cat results/results_eval_20260422_233310.json

cat: results/results_eval_20260422_233310.json: No such file or directory


Notes: as you can see in the output above, Qwen is repeating itself (probably because we need to use a proper chat template prompt or switch to a different model optimized for QA like flan T5), we can also try reducing the max_tokens if needed. The repeated keys in relevant_ids is not an issue as we convert it to a set in metrics.py